## Create correlation matrices

In [1]:

# Library imports
import numpy as np
import pickle
from nilearn.connectome import ConnectivityMeasure
from joblib import Parallel, delayed
import os
from itertools import combinations

# ----------------------------
# 1. Load data
# ----------------------------
print("Loading fMRI data and metadata...")
data = np.load('fMRI_data.npz')['data']
with open('metadata.pkl', 'rb') as f:
    meta = pickle.load(f)

data = data.astype(np.float32)
n_subjects_orig = data.shape[0]
print(f"Original data shape: {data.shape} ({n_subjects_orig} subjects)")

# ----------------------------
# 2. Remove subjects with NaNs
# ----------------------------
print("Checking for NaNs...")
nan_per_subject = np.isnan(data).any(axis=(1, 2))
n_bad = nan_per_subject.sum()

if n_bad > 0:
    print(f"⚠️  Found {n_bad} subjects with NaNs. Removing them.")
    good_mask = ~nan_per_subject
    data_clean = data[good_mask]
    subject_ids_clean = [meta['subject_ids'][i] for i in range(n_subjects_orig) if good_mask[i]]
    print(f"✅ Kept {data_clean.shape[0]} subjects.")
else:
    data_clean = data
    subject_ids_clean = meta['subject_ids']
    print("✅ No NaNs found.")

n_subjects_clean = data_clean.shape[0]
n_regions = data_clean.shape[2]
region_names = meta['region_names']

# ----------------------------
# 3. Generate FEATURE NAMES: all ROI pairs
# ----------------------------
print("Generating feature names (ROI pairs)...")
feature_names = [f"{r1}--{r2}" for r1, r2 in combinations(region_names, 2)]
expected_n_features = len(feature_names)
print(f"Total features (ROI pairs): {expected_n_features}")

# ----------------------------
# 4. Compute connectivity in parallel
# ----------------------------
def compute_connectivity(subject_data):
    cm = ConnectivityMeasure(
        kind='correlation',
        vectorize=True,
        discard_diagonal=True,
        standardize=True
    )
    return cm.fit_transform([subject_data])[0]

print(f"\nComputing correlation features using {os.cpu_count()} threads...")
correlation_features = Parallel(n_jobs=-1, verbose=10)(
    delayed(compute_connectivity)(data_clean[i])
    for i in range(n_subjects_clean)
)
correlation_features = np.asarray(correlation_features, dtype=np.float32)

assert correlation_features.shape == (n_subjects_clean, expected_n_features)
print(f"✅ Feature matrix shape: {correlation_features.shape}")

# ----------------------------
# 5. SAVE EVERYTHING — fully analysis-ready
# ----------------------------
output_path = 'correlation_connectivity_features_complete.npz'
np.savez_compressed(
    output_path,
    # Core data
    features=correlation_features,          # (n_subjects, n_features)
    subject_ids=subject_ids_clean,         # [str] — matches your phenotypic EIDs
    feature_names=feature_names,           # [str] — e.g., "R_V1_ROI--L_MT_ROI"
    region_names=region_names,             # [str] — original 414 ROIs
    # Metadata
    n_subjects=n_subjects_clean,
    n_regions=n_regions,
    connectivity_kind='correlation'
)

print(f"\n✅ FULLY SAVED to '{output_path}'")
print("Contents:")
print("  - 'features': connectivity values")
print("  - 'subject_ids': subject EIDs (for phenotypic merge)")
print("  - 'feature_names': human-readable ROI pair names")
print("  - 'region_names': original ROI list")
print("Done.")

Loading fMRI data and metadata...
Original data shape: (16382, 490, 414) (16382 subjects)
Checking for NaNs...
⚠️  Found 42 subjects with NaNs. Removing them.
✅ Kept 16340 subjects.
Generating feature names (ROI pairs)...
Total features (ROI pairs): 85491

Computing correlation features using 40 threads...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 40 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done  33 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:    1.5s
[Parallel(n_jobs=-1)]: Done  65 tasks      | elapsed:    1.5s
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:    1.5s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.19185067678209766s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done 101 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done 120 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done 162 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done 185 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.12466263771057129s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done 216 tasks      | elap

✅ Feature matrix shape: (16340, 85491)

✅ FULLY SAVED to 'correlation_connectivity_features_complete.npz'
Contents:
  - 'features': connectivity values
  - 'subject_ids': subject EIDs (for phenotypic merge)
  - 'feature_names': human-readable ROI pair names
  - 'region_names': original ROI list
Done.


In [2]:
import pandas as pd

# Read file
pheno_df = pd.read_csv('/home/jaizor/jaizor/xtra/notebooks/UKBB/data/csv/ukbb_fmri.csv')

pheno_df.head()

,eid,Sex,Chronic_Blood_Immune,Endocrine_Nutritional_Metabolic_Hypothyroidism,Endocrine_Nutritional_Metabolic_Hyperthyroidism,Endocrine_Nutritional_Metabolic_Diabetes,Endocrine_Nutritional_Metabolic_Other_Pancreatic,Endocrine_Nutritional_Metabolic_Other_Endocrine,Endocrine_Nutritional_Metabolic_Protein_Energy_Malnutrition,Endocrine_Nutritional_Metabolic_Vitamin_Deficiency,...,Chronic_Congenital,Chronic_Chromosomal,ICD_F32_Depressive_Episode,ICD_F33_Recurrent_Depressive,ICD_G20_Parkinsons,ICD_G21_Secondary_Parkinsonism,ICD_G40_Epilepsy,ICD_G41_Status_Epilepticus,ICD_F20_Schizophrenia,ICD_F31_Bipolar
0,1000097,Female,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1000276,Female,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1000333,Male,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1000541,Male,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1000553,Female,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
pheno_df.shape

(16382, 43)